# Badini TTS — inference for hayfa, bafreen & abdullah  (no kernel switch)

Runs on the **default "Python 3" kernel** — no cosyvoice kernel, no register, no refresh.
Cell 1 loads the **train_all fine-tuned model** (`/workspace/cosyvoice3_badini_all/model_ft`) once in a background worker inside the cosyvoice env and
sets up the three voices, each cloned from a clean 3–12 s prompt clip of that speaker (from the cleaned train split).
Then `say("text", voice="hayfa"|"bafreen"|"abdullah")` prints the text and plays the audio.

**Numbers and abbreviations are spoken in Badini:** the worker uses `text_frontend=False`, and `say()` first converts them to Badini words:
whole numbers, negatives (-234), decimals (34.567), currency (45.6$), `+` (345+278), dates (2026/11/23), times (8:45:35), Latin or Arabic-Indic digits,
and abbreviations (د. پ.د. م. ژ. ت. پ. ز، كم كگم …).
Number words follow your corrected Excel sheet; edit them at the top of cell 2 if needed.

Run cell 1 once (~30 s), then use the cells below. If the kernel restarts, re-run cell 1.

## 1 · Start the TTS engine  (run once — loads the model in the background)

In [1]:
import subprocess, os

CENV = "/workspace/envs/cosyvoice/bin/python"

worker = r"""
import sys, os, contextlib, wave
WORK="/workspace/cosyvoice3_badini_all"; SERVE=WORK+"/model_ft"; COSY="/workspace/CosyVoice"
INSTRUCT="You are a helpful assistant.<|endofprompt|>"
PROTO=sys.stdout
def emit(s): PROTO.write(s+"\n"); PROTO.flush()

def dur(p):
    try:
        with contextlib.closing(wave.open(p)) as w: return w.getnframes()/w.getframerate()
    except Exception: return None

def load_split(name):                       # cleaned Kaldi files written by the training notebook
    d=WORK+"/data/"+name; wav={}; txt={}; spk={}
    for l in open(d+"/wav.scp",encoding="utf-8"):  u,w=l.rstrip("\n").split(" ",1); wav[u]=w
    for l in open(d+"/text",encoding="utf-8"):     u,t=l.rstrip("\n").split(" ",1); txt[u]=t
    for l in open(d+"/utt2spk",encoding="utf-8"):  u,s=l.rstrip("\n").split(" ",1); spk[u]=s
    return {u:(wav[u],txt[u],spk[u]) for u in wav}

with contextlib.redirect_stdout(sys.stderr):
    import torch, torchaudio
    sys.path.append(COSY+"/third_party/Matcha-TTS"); sys.path.append(COSY)
    from cosyvoice.cli.cosyvoice import AutoModel
    cosy=AutoModel(model_dir=SERVE); SR=cosy.sample_rate
    tr=load_split("train")
    VOICES={}
    for s in sorted({v[2] for v in tr.values()}):          # first 3-12 s train clip of each speaker
        for u in sorted(tr):
            w,t,sp=tr[u]
            if sp!=s: continue
            d=dur(w) if os.path.exists(w) else None
            if d and 3.0<=d<=12.0: VOICES[s]=(w,t); break
emit("READY\t"+str(SR)+"\t"+"\t".join(k+"="+VOICES[k][0] for k in sorted(VOICES)))
for line in sys.stdin:
    line=line.rstrip("\n")
    if not line.strip(): continue
    try:
        voice,out,text=line.split("\t",2)
        pw,pt=VOICES[voice]
        with contextlib.redirect_stdout(sys.stderr):
            for j in cosy.inference_zero_shot(text, INSTRUCT+pt, pw, stream=False, text_frontend=False):
                torchaudio.save(out, j["tts_speech"], SR)
        emit("OK\t"+out)
    except Exception as e:
        emit("ERR\t"+str(e).replace("\n"," "))
"""
open("/workspace/tts_worker.py","w").write(worker)

env={**os.environ, "HF_HUB_ENABLE_HF_TRANSFER":"0"}
proc=subprocess.Popen([CENV,"/workspace/tts_worker.py"], stdin=subprocess.PIPE,
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, bufsize=1, env=env)
print("loading model in background (~30s)...")
SR=None
for line in proc.stdout:
    if line.startswith("READY"):
        parts=line.strip().split("\t"); SR=int(parts[1])
        print("ready — sr", SR)
        for p in parts[2:]: print("  prompt:", p)
        assert len(parts[2:])==3, "expected 3 voices (abdullah, bafreen, hayfa)"
        break


loading model in background (~30s)...
ready — sr 24000
  prompt: abdullah=/workspace/Data24_09_2026/clips/2_44100_a1.wav
  prompt: bafreen=/workspace/Data24_09_2026/clips/1_44100_a100.wav
  prompt: hayfa=/workspace/Data24_09_2026/clips/hayfa98_44100_a1.wav


## 2 · The `say` function

In [2]:
import IPython.display as ipd, itertools, os, re
OUT="/workspace/cosyvoice3_badini_all/tts_out"; os.makedirs(OUT, exist_ok=True)
_c=itertools.count()

# ---- digits -> Badini words (spelling taken from the user's corrected Excel sheet) ----
ONES  = {1:"ئێک",2:"دو",3:"سێ",4:"چار",5:"پێنج",6:"شەش",7:"حەفت",8:"هەشت",9:"نەه",
         10:"دەه",11:"یازدە",12:"دوازدە",13:"سیزدە",14:"چاردە",15:"پازدە",
         16:"شازدە",17:"حەڤدە",18:"هەژدە",19:"نوزدە"}
TENS  = {2:"بیست",3:"سیه",4:"چل",5:"پێنجی",6:"شێست",7:"حەفتێ",8:"هەشتا",9:"نوت"}
HUND, THOU, ZERO, AND = "سەد", "هزار", "سفر", " و "
POINT, HOUR, MINUTE, SECOND = "پوینت", "دەمژمێر", "خولەک", "چرکە"   # HOUR is said BEFORE the hour number
PLUS, MINUS = "کومکرنی", "ژێبرن"
DAY, MONTH, YEAR = "رۆژا", "هەیڤا", "سالا"
CURR = {"$": ("دولار", "سەنت"), "€": ("یۆرۆ", "سەنت")}   # symbol -> (main unit, cents)
D = "0-9٠-٩۰-۹"

def _below100(r):
    if r < 20: return ONES[r]
    t, u = divmod(r, 10)
    return TENS[t] + (AND + ONES[u] if u else "")

def _below1000(n):
    h, r = divmod(n, 100)
    parts = []
    if h: parts.append(HUND if h == 1 else ONES[h] + " " + HUND)
    if r: parts.append(_below100(r))
    return AND.join(parts)

def num2words_badini(n):
    if n == 0: return ZERO
    th, r = divmod(n, 1000)
    parts = []
    if th: parts.append(THOU if th == 1 else _below1000(th) + " " + THOU)
    if r:  parts.append(_below1000(r))
    return AND.join(parts)

_ASCII = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")

def _int(s):                       # digit string -> words (0-999999; longer numbers / 007 codes stay as digits)
    s = s.translate(_ASCII)
    if len(s) > 1 and s[0] == "0": return s
    n = int(s)
    return num2words_badini(n) if n <= 999999 else s

def _frac(s):                      # fraction read as a number:  23 -> بیست و سێ ;  05 -> سفر پێنج
    s = s.translate(_ASCII); z = len(s) - len(s.lstrip("0")); rest = s.lstrip("0")
    return " ".join([ZERO] * z + ([_int(rest)] if rest else []))

def _num(s):                       # 34.567 / 45.6 / 12
    s = s.translate(_ASCII)
    if "." in s:
        i, f = s.split(".", 1)
        return f"{_int(i)} {POINT} {_frac(f)}"
    return _int(s)

def _money(s, sym):                # 45.6$ -> چل و پێنج دولار و شەش سەنت
    main, cent = CURR[sym]; s = s.translate(_ASCII)
    i, _, f = s.partition(".")
    out = f"{_int(i)} {main}"
    if f.strip("0"): out += f"{AND}{_int(f.lstrip('0'))} {cent}"
    return out

def _time(m):                      # 21:45:30 -> دەمژمێر بیست و ئێک و چل و پێنج خولەک و سیه چرکە
    h, mi, se = m.group(1), m.group(2), m.group(3)
    out = [f"{HOUR} {_int(h)}"]
    if int(mi.translate(_ASCII)): out.append(f"{_int(mi)} {MINUTE}")
    if se and int(se.translate(_ASCII)): out.append(f"{_int(se)} {SECOND}")
    return AND.join(out)

def _date(m):
    g = m.groups()
    if len(g[0].translate(_ASCII)) == 4: y, mo, d = g[0], g[1], g[2]     # 2026/11/23
    else:                                d, mo, y = g[0], g[1], g[2]     # 23/11/2026
    before = m.string[:m.start()].rstrip()
    day = "" if re.search(r"[ڕر]ۆژا$", before) else DAY + " "     # text already says "ڕۆژا" -> don't repeat it
    return f"{day}{_int(d)}، {MONTH} {_int(mo)}، {YEAR} {_int(y)}"

# ---- abbreviations (from your Excel table) -> full words, applied before the digit conversion ----
KURDISH_LETTERS = True    # write the expanded words with Kurdish ک / ی instead of Arabic ك / ي (check with the data check cell)
ABBR_TITLES = {            # only at the start of a word, written with a dot
    "پ.د.": "پرۆفیسۆر دكتۆر", "م.د.": "مامۆستا دكتۆر", "م.ب.": "مۆبایل", "ص.ب.": "سندوقا پۆستێ",
    "د.": "دكتۆر", "پ.": "پرۆفیسۆر", "م.": "مامۆستا", "ژ.": "ژمارە", "ت.": "تەلەفۆن"}
ABBR_UNITS = {             # only directly after a number:  5 كم -> پێنج كیلۆمەتر
    "كگم": "كیلۆگرام", "ملم": "ملیمەتر", "كم": "كیلۆمەتر", "سم": "سانتیمەتر", "گم": "گرام", "م": "مەتر"}
ABBR_RELIGIOUS = {"(ص)": "(سەڵا الله عەلەیهی وە سەلەم)", "(ع.س.)": "(عەلەیهی ئەلسلام)"}

def _kd(s):
    return s.replace("ك", "ک").replace("ي", "ی") if KURDISH_LETTERS else s
def _kaf(s):                                   # match Arabic ك and Kurdish ک in the input
    return s.replace("ك", "[كک]")

def expand_abbreviations(text):
    for k, v in ABBR_RELIGIOUS.items():
        text = text.replace(k, _kd(v))
    text = re.sub(r"(?<![\w.])پ\.\s*ز\.?(?![\w])", "پێش زایین", text)   # پ. ز = BC (before the Christian era), not "professor"
    t = "|".join(re.escape(_kaf(k)).replace(r"\[كک\]", "[كک]") for k in sorted(ABBR_TITLES, key=len, reverse=True))
    def title(m):
        key = m.group(0).replace("ک", "ك")
        nxt = m.string[m.end():m.end()+1]
        return _kd(ABBR_TITLES[key]) + ("" if (nxt == "" or nxt.isspace()) else " ")   # keep a space in "د.محمد"
    text = re.sub(rf"(?<![\w.])(?:{t})(?=\s*\S)", title, text)
    u = "|".join(_kaf(k) for k in sorted(ABBR_UNITS, key=len, reverse=True))
    def unit(m):
        before = m.string[:m.start()]
        if re.search(r":[0-9٠-٩۰-۹]{2}\s*$", before):            # 10:30 م  -> a time of day, leave it
            return m.group(0)
        if m.group(1) == "م" and (re.search(r"(?<![0-9٠-٩۰-۹])[0-9٠-٩۰-۹]{4}\s*$", before)      # 2024 م  -> year marker
                                  or re.search(r"[0-9٠-٩۰-۹]+[/\\][0-9٠-٩۰-۹]+[/\\][0-9٠-٩۰-۹]+\s*$", before)):  # 23/12/2024 م -> date marker
            return m.group(0)
        return " " + _kd(ABBR_UNITS[m.group(1).replace("ک", "ك")])
    text = re.sub(rf"(?<=[0-9٠-٩۰-۹])\s*({u})(?![\w])", unit, text)
    return re.sub(r"[ \t]{2,}", " ", text)

def digits_to_words(text):
    text = expand_abbreviations(text)             # abbreviations -> words first
    # 0) Arabic decimal/thousands marks -> ASCII; minus; plus
    text = text.replace("٫", ".").replace("٬", ",")
    text = re.sub(rf"(?<!\w)[-−](?=[{D}])", f"{MINUS} ", text)          # -234 / -56.23 (not 2020-2024)
    text = re.sub(rf"(?<=[{D}])\s*\+\s*(?=[{D}])", f" {PLUS} ", text)       # 234+ 78
    # 1) dates  2026/11/23, 23/11/2026 or 10\7\2018 (backslash form used in your data)
    text = re.sub(rf"(?<![{D}])([{D}]{{4}}|[{D}]{{1,2}})[/\\]([{D}]{{1,2}})[/\\]([{D}]{{4}}|[{D}]{{1,2}})(?![{D}])", _date, text)
    # 2) time  21:45:30 / 21:45
    text = re.sub(rf"(?<![{D}])([{D}]{{1,2}}):([{D}]{{2}})(?::([{D}]{{2}}))?(?![{D}])", _time, text)
    # 3) thousands commas  1,250 -> 1250
    text = re.sub(rf"(?<=[{D}]),(?=[{D}]{{3}}(?![{D}]))", "", text)
    # 4) currency  45.6$  /  $45.6
    sym = "".join(re.escape(k) for k in CURR)
    text = re.sub(rf"([{D}]+(?:\.[{D}]+)?)\s*([{sym}])", lambda m: _money(m.group(1), m.group(2)), text)
    text = re.sub(rf"([{sym}])\s*([{D}]+(?:\.[{D}]+)?)", lambda m: _money(m.group(2), m.group(1)), text)
    # 5) decimals and plain numbers
    return re.sub(rf"[{D}]+(?:\.[{D}]+)?", lambda m: _num(m.group(0)), text)

def say(text, voice="hayfa"):
    assert voice in ("hayfa","bafreen","abdullah"), "voice must be 'hayfa', 'bafreen' or 'abdullah'"
    text = digits_to_words(text)          # numbers/dates/times/etc. -> Badini words
    print(f"[{voice}]  {text}")
    o=f"{OUT}/{voice}_{next(_c):03d}.wav"
    proc.stdin.write(f"{voice}\t{o}\t{text.strip()}\n"); proc.stdin.flush()
    r=proc.stdout.readline().strip()
    if r.startswith("OK"): return ipd.Audio(o)
    print("error:", r)


## 3 · hayfa voice — edit the text

In [3]:
say("ئەز ئامادە نینم ببمە دادۆکێ بچویکێ هیچ فەلیتەکی.", voice="hayfa")

[hayfa]  ئەز ئامادە نینم ببمە دادۆکێ بچویکێ هیچ فەلیتەکی.


## 4 · bafreen voice — edit the text

In [4]:
say("چیڕوکێن وی ڕوناهیێ دهاڤێژن سەر سەرڕەقی و بەلەنگازیا ژیانێ.", voice="bafreen")

[bafreen]  چیڕوکێن وی ڕوناهیێ دهاڤێژن سەر سەرڕەقی و بەلەنگازیا ژیانێ.


## 5 · abdullah voice — edit the text

In [5]:
say("ئەز ئامادە نینم ببمە دادۆکێ بچویکێ هیچ فەلیتەکی.", voice="abdullah")

[abdullah]  ئەز ئامادە نینم ببمە دادۆکێ بچویکێ هیچ فەلیتەکی.


## 6 · (optional) same sentence in all three voices

In [6]:
from IPython.display import display
TXT = "سلاڤ، ئەڤ دەنگە ب زمانێ کوردی یێ بادینی هاتیە دروستکرن."
display(say(TXT, voice="hayfa"))
display(say(TXT, voice="bafreen"))
display(say(TXT, voice="abdullah"))

[hayfa]  سلاڤ، ئەڤ دەنگە ب زمانێ کوردی یێ بادینی هاتیە دروستکرن.


[bafreen]  سلاڤ، ئەڤ دەنگە ب زمانێ کوردی یێ بادینی هاتیە دروستکرن.


[abdullah]  سلاڤ، ئەڤ دەنگە ب زمانێ کوردی یێ بادینی هاتیە دروستکرن.


## 7 · Test these two texts in all three voices

## 8 · Number and abbreviation test — all three voices

In [12]:
for s in ["34.567", "45.6$", "21:45:30", "2026/12/23", "10:00", "1,250", "234+ 78", "-56.23", "-234", "د. محمد", "پ.د. ئیبراهیم ژ. 5", "5 كم", "پ. ز", "26", "101", "2024", "1999", "12345"]:
    print(s, "->", digits_to_words(s))
from IPython.display import display
T = "234+ 78 و -56.23 و 45.6$ د ساعەت 21:45:30 دا، رۆژا 2026/12/23 بوو، 1,250 کەس."

T=" ئاماژە ب بریارا جڤاتا زانکۆیێ ژمارە (٥ ئاسایی) ل رێکەفتی ٨/١/٢٠٢٥، سەروکاتیا زانکۆیا دهۆک رادگەهینیت کو دێ رابیت ب راگەهاندنا پوستێ رێڤەبەرێ سەنتەرێ ڤەکولینێن سازی ل سەروکاتیا زانکۆیا دهۆک ، هەر ماموستایەکێ حەز هەبیت مافێ پێشکێشکرنێ یێ هەی بو پوستێن دیارکری برێیا (لینکێ) پوستێن هەڤپێچ کری."
print("— hayfa —");   display(say(T, voice="hayfa"))
print("— bafreen —"); display(say(T, voice="bafreen"))
print("— abdullah —"); display(say(T, voice="abdullah"))


34.567 -> سیه و چار پوینت پێنج سەد و شێست و حەفت
45.6$ -> چل و پێنج دولار و شەش سەنت
21:45:30 -> دەمژمێر بیست و ئێک و چل و پێنج خولەک و سیه چرکە
2026/12/23 -> رۆژا بیست و سێ، هەیڤا دوازدە، سالا دو هزار و بیست و شەش
10:00 -> دەمژمێر دەه
1,250 -> هزار و دو سەد و پێنجی
234+ 78 -> دو سەد و سیه و چار کومکرنی حەفتێ و هەشت
-56.23 -> ژێبرن پێنجی و شەش پوینت بیست و سێ
-234 -> ژێبرن دو سەد و سیه و چار
د. محمد -> دکتۆر محمد
پ.د. ئیبراهیم ژ. 5 -> پرۆفیسۆر دکتۆر ئیبراهیم ژمارە پێنج
5 كم -> پێنج کیلۆمەتر
پ. ز -> پێش زایین
26 -> بیست و شەش
101 -> سەد و ئێک
2024 -> دو هزار و بیست و چار
1999 -> هزار و نەه سەد و نوت و نەه
12345 -> دوازدە هزار و سێ سەد و چل و پێنج
— hayfa —
[hayfa]   ئاماژە ب بریارا جڤاتا زانکۆیێ ژمارە (پێنج ئاسایی) ل رێکەفتی رۆژا هەشت، هەیڤا ئێک، سالا دو هزار و بیست و پێنج، سەروکاتیا زانکۆیا دهۆک رادگەهینیت کو دێ رابیت ب راگەهاندنا پوستێ رێڤەبەرێ سەنتەرێ ڤەکولینێن سازی ل سەروکاتیا زانکۆیا دهۆک ، هەر ماموستایەکێ حەز هەبیت مافێ پێشکێشکرنێ یێ هەی بو پوستێن دیارکری برێیا (لینکێ) پوستێن هەڤپ

— bafreen —
[bafreen]   ئاماژە ب بریارا جڤاتا زانکۆیێ ژمارە (پێنج ئاسایی) ل رێکەفتی رۆژا هەشت، هەیڤا ئێک، سالا دو هزار و بیست و پێنج، سەروکاتیا زانکۆیا دهۆک رادگەهینیت کو دێ رابیت ب راگەهاندنا پوستێ رێڤەبەرێ سەنتەرێ ڤەکولینێن سازی ل سەروکاتیا زانکۆیا دهۆک ، هەر ماموستایەکێ حەز هەبیت مافێ پێشکێشکرنێ یێ هەی بو پوستێن دیارکری برێیا (لینکێ) پوستێن هەڤپێچ کری.


— abdullah —
[abdullah]   ئاماژە ب بریارا جڤاتا زانکۆیێ ژمارە (پێنج ئاسایی) ل رێکەفتی رۆژا هەشت، هەیڤا ئێک، سالا دو هزار و بیست و پێنج، سەروکاتیا زانکۆیا دهۆک رادگەهینیت کو دێ رابیت ب راگەهاندنا پوستێ رێڤەبەرێ سەنتەرێ ڤەکولینێن سازی ل سەروکاتیا زانکۆیا دهۆک ، هەر ماموستایەکێ حەز هەبیت مافێ پێشکێشکرنێ یێ هەی بو پوستێن دیارکری برێیا (لینکێ) پوستێن هەڤپێچ کری.
